# AI Job Fit Analyzer – Jupyter Notebook
**Modules:** Resume Parsing, NER Skill Extraction, Sentiment Analysis, Match Score (Linear Regression), Job Role Classification (Logistic Regression)

In [1]:
# Required Libraries
import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import LabelEncoder
from transformers import pipeline

In [2]:
# Sample Resumes, Job Descriptions, and Cover Letters
resumes = [
    "Experienced Python developer with Django and Flask skills.",
    "Creative graphic designer skilled in Adobe Photoshop and Illustrator.",
    "Data analyst familiar with SQL, Excel, and Tableau.",
    "Backend engineer with Node.js and MongoDB experience.",
    "Business analyst with strong presentation and communication skills."
]

job_descriptions = [
    "Looking for a Python backend developer with experience in Django and REST APIs.",
    "Seeking a designer with Photoshop and creative design abilities.",
    "Hiring a data analyst with SQL and dashboarding tools like Tableau.",
    "Need a backend engineer skilled in Node.js and databases like MongoDB.",
    "Looking for an analyst with great communication and stakeholder skills."
]

cover_letters = [
    "I am passionate about backend development and love building REST APIs.",
    "Design is my passion, and I enjoy creating stunning visual content.",
    "I thrive on analyzing data and presenting insights with clarity.",
    "My focus is always performance and scalability in backend systems.",
    "I communicate effectively and translate business needs into analytics solutions."
]

job_roles = ["Developer", "Designer", "Analyst", "Developer", "Analyst"]

In [3]:
# Step 1: Skill Extraction using keywords
keywords = ["python", "django", "flask", "photoshop", "illustrator", "sql", "excel", "tableau",
            "node.js", "mongodb", "presentation", "communication", "rest", "design"]

def extract_features(texts):
    return np.array([[1 if kw in text.lower() else 0 for kw in keywords] for text in texts])

resume_features = extract_features(resumes)
job_features = extract_features(job_descriptions)

In [4]:
# Step 2: Match Score Prediction using Linear Regression
X_match = np.abs(resume_features - job_features)
y_match = np.random.randint(60, 100, size=len(resumes))
reg_model = LinearRegression()
reg_model.fit(X_match, y_match)
predicted_scores = reg_model.predict(X_match)

In [5]:
# Step 3: Sentiment Analysis using Hugging Face
sentiment_analyzer = pipeline("sentiment-analysis")
sentiments = [sentiment_analyzer(cl)[0] for cl in cover_letters]
sentiment_labels = [s["label"] for s in sentiments]
sentiment_scores = [s["score"] for s in sentiments]

No model was supplied, defaulted to distilbert/distilbert-base-uncased-finetuned-sst-2-english and revision 714eb0f (https://huggingface.co/distilbert/distilbert-base-uncased-finetuned-sst-2-english).
Using a pipeline without specifying a model name and revision in production is not recommended.


config.json:   0%|          | 0.00/629 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

Device set to use cpu


In [ ]:
# Step 4: Job Role Classification
label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(job_roles)
X_train, X_test, y_train, y_test = train_test_split(resume_features, y_encoded, test_size=0.2, random_state=42)
clf = LogisticRegression()
clf.fit(X_train, y_train)
y_pred = clf.predict(X_test)
classification_accuracy = accuracy_score(y_test, y_pred)

In [ ]:
# Step 5: Combine Results
results_df = pd.DataFrame({
    "Resume": resumes,
    "Predicted_Match_Score": predicted_scores.astype(int),
    "Sentiment_Label": sentiment_labels,
    "Sentiment_Score": np.round(sentiment_scores, 2),
    "True_Role": job_roles,
    "Predicted_Role": label_encoder.inverse_transform(clf.predict(resume_features))
})
results_df["Match_Quality"] = results_df["Predicted_Match_Score"].apply(lambda x: "Good" if x > 80 else "Average")
results_df